# 03 SQL analysis with DuckDB

Answers the project's public questions with SQL, straight on the Parquet tables from notebook 02. DuckDB runs inside the notebook, so there is no database server to set up, and the same queries would run on any SQL warehouse with small changes.

**Questions answered here**

| Query | Question |
| --- | --- |
| q01 | Is every area and year complete, and is the population the same in every year? |
| q02 | How much PM2.5 does the average person in Bangladesh breathe each year? |
| q03 | How much has exposure risen, comparing the first and last five years? |
| q04 | Which divisions are worst, and has the order changed? |
| q05 | Which districts are the most and least polluted now? |
| q06 | Which upazilas are the hotspots, and how many people live there? |
| q07 | How many people live in each WHO band, year by year? |
| q08 | How many people in each division live above interim target 1 (35 µg/m³)? |
| q09 | Which months are worst, and does the peak month differ by division? |
| q10 | Is the dry season getting worse faster than the monsoon? |
| q11 | Does the 2018 jump change the national trend? |
| q12 | Where does population weighting matter most? |

**Why five-year periods:** single years move a lot with weather (and 2018 is unusually high in this release, see notebook 02). Comparing the mean of the first five years with the mean of the last five gives a steadier picture of change. The formal trend tests (Mann-Kendall, Sen's slope) are done in R.

**Inputs:** `data/processed/` (pm25_unit_year, pm25_unit_month, pm25_who_bands, bgd_admin.gpkg)

**Outputs:** one CSV per query in `outputs/sql_results/`

In [1]:
import duckdb, pandas as pd, geopandas as gpd, pathlib

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = pathlib.Path("/content/drive/MyDrive/Bangladesh Air Pollution Project")
except ImportError:
    ROOT = pathlib.Path.cwd().parent

PROC, OUT = ROOT / "data" / "processed", ROOT / "outputs" / "sql_results"
assert (PROC / "pm25_unit_year.parquet").exists(), "run notebook 02 first"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_rows", 200, "display.max_columns", 40, "display.width", 250)
con = duckdb.connect()

def run(name, sql):
    result = con.sql(sql).df()
    whole = [c for c in result.select_dtypes("float") if (result[c].dropna() % 1 == 0).all()]
    result[whole] = result[whole].astype("Int64")
    result.to_csv(OUT / f"{name}.csv", index=False)
    return result

Mounted at /content/drive


## Views

The Parquet files are exposed as views, so every query reads like a query on ordinary tables. Things added once here instead of in every query:
- `areas`: every upazila, district and division with its parent codes and area in km², from the boundary file (geometry dropped)
- `exposure`: yearly table joined to `areas`, so each row knows its district and division
- `monthly`: adds `year` and `month`, and `dry_season_year`, which puts December with the following January and February so one dry season is not split across two years
- `periods`: the first and last five-year windows, worked out from the data so they stay right if more years are added

In [2]:
areas = pd.concat([gpd.read_file(PROC / "bgd_admin.gpkg", layer=lvl).drop(columns="geometry").assign(level=lvl) for lvl in ("division", "district", "upazila")], ignore_index=True)
con.register("areas_raw", areas)

views = f"""
CREATE OR REPLACE VIEW areas AS
SELECT
    level                                                           AS level,
    pcode                                                           AS pcode,
    name                                                            AS name,
    COALESCE(district_pcode, CASE WHEN level = 'district' THEN pcode END)
                                                                    AS district_pcode,
    COALESCE(division_pcode, CASE WHEN level = 'division' THEN pcode END)
                                                                    AS division_pcode,
    area_km2                                                        AS area_km2
FROM areas_raw;

CREATE OR REPLACE VIEW exposure AS
SELECT
    e.level                                                         AS level,
    e.pcode                                                         AS pcode,
    e.name                                                          AS name,
    CAST(e.year AS INTEGER)                                         AS year,
    e.pop                                                           AS population,
    e.pm25_popw                                                     AS pm25_popw,
    e.pm25_area                                                     AS pm25_area,
    a.district_pcode                                                AS district_pcode,
    a.division_pcode                                                AS division_pcode,
    a.area_km2                                                      AS area_km2
FROM read_parquet('{PROC / "pm25_unit_year.parquet"}') AS e
LEFT JOIN areas AS a
  ON a.pcode = e.pcode;

CREATE OR REPLACE VIEW monthly AS
SELECT
    level                                                           AS level,
    pcode                                                           AS pcode,
    name                                                            AS name,
    CAST(date AS DATE)                                              AS date,
    YEAR(date)                                                      AS year,
    MONTH(date)                                                     AS month,
    CASE WHEN MONTH(date) = 12 THEN YEAR(date) + 1 ELSE YEAR(date) END
                                                                    AS dry_season_year,
    pm25_popw                                                       AS pm25_popw
FROM read_parquet('{PROC / "pm25_unit_month.parquet"}');

CREATE OR REPLACE VIEW who_bands AS
SELECT
    level                                                           AS level,
    pcode                                                           AS pcode,
    name                                                            AS name,
    CAST(year AS INTEGER)                                           AS year,
    band                                                            AS band,
    people                                                          AS people,
    share                                                           AS share
FROM read_parquet('{PROC / "pm25_who_bands.parquet"}');

CREATE OR REPLACE VIEW periods AS
SELECT
    MIN(year)                                                       AS first_year,
    MIN(year) + 4                                                   AS first_period_end,
    MAX(year) - 4                                                   AS last_period_start,
    MAX(year)                                                       AS last_year
FROM exposure;
"""
con.sql(views)
con.sql("SELECT * FROM periods").df()

,first_year,first_period_end,last_period_start,last_year
0,1998,2002,2020,2024


## q01. Is every area and year complete?

Before any result is read: how many areas and years each level has, whether any value is missing, and whether the population behind the weighted means is the same in every year (it should be, because one population map is used for all years).

*SQL used: `COUNT(DISTINCT)`, `COUNT(*) FILTER`, a population range check.*

In [3]:
q01_coverage = """
SELECT
    level                                                           AS level,
    COUNT(DISTINCT pcode)                                           AS areas,
    COUNT(DISTINCT year)                                            AS years,
    MIN(year)                                                       AS first_year,
    MAX(year)                                                       AS last_year,
    COUNT(*)                                                        AS rows,
    COUNT(*) FILTER (WHERE pm25_popw IS NULL)                       AS missing_popw,
    COUNT(*) FILTER (WHERE pm25_area IS NULL)                       AS missing_area_mean,
    ROUND(SUM(population) FILTER (WHERE year = (SELECT first_year FROM periods)))
                                                                    AS population_first_year,
    ROUND(SUM(population) FILTER (WHERE year = (SELECT last_year FROM periods)))
                                                                    AS population_last_year
FROM exposure
GROUP BY level
ORDER BY areas
"""
run("q01_coverage", q01_coverage)

,level,areas,years,first_year,last_year,rows,missing_popw,missing_area_mean,population_first_year,population_last_year
0,national,1,27,1998,2024,27,0,0,168343283,168343283
1,division,8,27,1998,2024,216,0,0,168343283,168343283
2,district,64,27,1998,2024,1728,0,0,168343283,168343283
3,upazila,507,27,1998,2024,13689,0,0,168343283,168343283


## q02. How much PM2.5 does the average person breathe each year?

The national population-weighted mean, the simple area mean, how many times the WHO guideline (5 µg/m³) that is, the change from the previous year, and a centred three-year average that smooths out single unusual years.

*SQL used: `LAG` for year-on-year change, `AVG() OVER` with a `ROWS BETWEEN` frame for the three-year average.*

In [4]:
q02_national_yearly = """
SELECT
    year                                                            AS year,
    ROUND(pm25_popw, 1)                                             AS pm25_popw,
    ROUND(pm25_area, 1)                                             AS pm25_area,
    ROUND(pm25_popw / 5, 1)                                         AS times_who_guideline,
    ROUND(pm25_popw - LAG(pm25_popw) OVER (ORDER BY year), 1)       AS change_vs_previous_year,
    ROUND(AVG(pm25_popw) OVER (ORDER BY year
                               ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING), 1)
                                                                    AS centred_3yr_mean
FROM exposure
WHERE level = 'national'
ORDER BY year
"""
run("q02_national_yearly", q02_national_yearly)

,year,pm25_popw,pm25_area,times_who_guideline,change_vs_previous_year,centred_3yr_mean
0,1998,39.2,36.2,7.8,NaN,40.6
1,1999,41.9,38.4,8.4,2.7,44.4
2,2000,51.9,47.3,10.4,10.0,47.1
3,2001,47.6,43.9,9.5,-4.4,49.8
4,2002,49.8,45.6,10.0,2.2,50.6
5,2003,54.6,50.2,10.9,4.8,52.2
6,2004,52.3,48.5,10.5,-2.2,53.3
7,2005,52.9,48.1,10.6,0.6,55.4
8,2006,60.9,55.7,12.2,7.9,56.7
9,2007,56.3,52.4,11.3,-4.6,58.2


## q03. How much has exposure risen?

Mean of the first five years against the mean of the last five, for the country and each division, as an absolute and a percentage change.

*SQL used: conditional averages with `AVG() FILTER`, a `CROSS JOIN` to the one-row `periods` view.*

In [5]:
q03_period_change = """
WITH by_period AS (
    SELECT
        e.level                                                     AS level,
        e.name                                                      AS area,
        AVG(e.pm25_popw) FILTER (WHERE e.year <= p.first_period_end)
                                                                    AS first_5yr,
        AVG(e.pm25_popw) FILTER (WHERE e.year >= p.last_period_start)
                                                                    AS last_5yr
    FROM exposure AS e
    CROSS JOIN periods AS p
    WHERE e.level IN ('national', 'division')
    GROUP BY e.level, e.name
)
SELECT
    level                                                           AS level,
    area                                                            AS area,
    ROUND(first_5yr, 1)                                             AS first_5yr_mean,
    ROUND(last_5yr, 1)                                              AS last_5yr_mean,
    ROUND(last_5yr - first_5yr, 1)                                  AS change,
    ROUND(100.0 * (last_5yr / first_5yr - 1), 1)                    AS change_pct
FROM by_period
ORDER BY level DESC, change DESC
"""
run("q03_period_change", q03_period_change)

,level,area,first_5yr_mean,last_5yr_mean,change,change_pct
0,national,Bangladesh,46.1,67.6,21.6,46.8
1,division,Dhaka,54.4,79.8,25.3,46.5
2,division,Khulna,41.5,64.2,22.7,54.6
3,division,Rajshahi,43.8,65.9,22.1,50.4
4,division,Rangpur,43.5,64.8,21.3,48.9
5,division,Chattogram,46.8,66.3,19.5,41.8
6,division,Mymensingh,41.4,60.2,18.9,45.6
7,division,Sylhet,27.6,44.8,17.2,62.2
8,division,Barishal,50.2,67.1,16.9,33.7


## q04. Which divisions are worst, and has the order changed?

Divisions ranked by five-year mean exposure in the first and last periods, and how many places each moved.

*SQL used: `RANK() OVER` on two aggregates in the same query.*

In [6]:
q04_division_ranking = """
WITH by_period AS (
    SELECT
        e.name                                                      AS division,
        AVG(e.pm25_popw) FILTER (WHERE e.year <= p.first_period_end)
                                                                    AS first_5yr,
        AVG(e.pm25_popw) FILTER (WHERE e.year >= p.last_period_start)
                                                                    AS last_5yr,
        MAX(e.population)                                           AS population
    FROM exposure AS e
    CROSS JOIN periods AS p
    WHERE e.level = 'division'
    GROUP BY e.name
)
SELECT
    division                                                        AS division,
    ROUND(population / 1e6, 1)                                      AS population_m,
    ROUND(first_5yr, 1)                                             AS first_5yr_mean,
    RANK() OVER (ORDER BY first_5yr DESC)                           AS rank_first,
    ROUND(last_5yr, 1)                                              AS last_5yr_mean,
    RANK() OVER (ORDER BY last_5yr DESC)                            AS rank_last,
    RANK() OVER (ORDER BY first_5yr DESC) - RANK() OVER (ORDER BY last_5yr DESC)
                                                                    AS places_moved_up
FROM by_period
ORDER BY rank_last
"""
run("q04_division_ranking", q04_division_ranking)

,division,population_m,first_5yr_mean,rank_first,last_5yr_mean,rank_last,places_moved_up
0,Dhaka,45.1,54.4,1,79.8,1,0
1,Barishal,9.3,50.2,2,67.1,2,0
2,Chattogram,33.8,46.8,3,66.3,3,0
3,Rajshahi,20.7,43.8,4,65.9,4,0
4,Rangpur,17.9,43.5,5,64.8,5,0
5,Khulna,17.8,41.5,6,64.2,6,0
6,Mymensingh,12.5,41.4,7,60.2,7,0
7,Sylhet,11.2,27.6,8,44.8,8,0


## q05. Which districts are the most and least polluted now?

The ten highest and ten lowest districts by last-five-year mean, with their division and population.

*SQL used: `QUALIFY` to keep the top and bottom ten without a subquery.*

In [7]:
q05_district_extremes = """
WITH recent AS (
    SELECT
        e.name                                                      AS district,
        d.name                                                      AS division,
        MAX(e.population)                                           AS population,
        AVG(e.pm25_popw)                                            AS last_5yr_mean
    FROM exposure AS e
    CROSS JOIN periods AS p
    JOIN areas AS d
      ON d.pcode = e.division_pcode
    WHERE e.level = 'district'
      AND e.year >= p.last_period_start
    GROUP BY e.name, d.name
)
SELECT
    CASE WHEN RANK() OVER (ORDER BY last_5yr_mean DESC) <= 10 THEN 'highest' ELSE 'lowest' END
                                                                    AS group_name,
    RANK() OVER (ORDER BY last_5yr_mean DESC)                       AS rank_highest,
    district                                                        AS district,
    division                                                        AS division,
    ROUND(population / 1e6, 2)                                      AS population_m,
    ROUND(last_5yr_mean, 1)                                         AS last_5yr_mean
FROM recent
QUALIFY RANK() OVER (ORDER BY last_5yr_mean DESC) <= 10
     OR RANK() OVER (ORDER BY last_5yr_mean ASC) <= 10
ORDER BY rank_highest
"""
run("q05_district_extremes", q05_district_extremes)

,group_name,rank_highest,district,division,population_m,last_5yr_mean
0,highest,1,Dhaka,Dhaka,14.90,83.9
1,highest,2,Narayanganj,Dhaka,3.97,82.5
2,highest,3,Gazipur,Dhaka,5.44,82.3
3,highest,4,Manikganj,Dhaka,1.57,82.2
4,highest,5,Munshiganj,Dhaka,1.71,81.5
5,highest,6,Narsingdi,Dhaka,2.68,79.8
6,highest,7,Madaripur,Dhaka,1.35,78.5
7,highest,8,Chandpur,Chattogram,2.65,77.0
8,highest,9,Faridpur,Dhaka,2.21,76.9
9,highest,10,Shariatpur,Dhaka,1.27,76.8


## q06. Which upazilas are the hotspots?

The 15 upazilas with the highest last-five-year mean, with district, division, population and how much they rose since the first five years. Hotspot testing (LISA, Gi*) comes later in R; this is the plain ranking.

*SQL used: two joins to `areas` for parent names, a `LIMIT` on an ordered aggregate.*

In [8]:
q06_upazila_hotspots = """
WITH by_period AS (
    SELECT
        e.pcode                                                     AS pcode,
        e.name                                                      AS upazila,
        e.district_pcode                                            AS district_pcode,
        e.division_pcode                                            AS division_pcode,
        MAX(e.population)                                           AS population,
        AVG(e.pm25_popw) FILTER (WHERE e.year <= p.first_period_end)
                                                                    AS first_5yr,
        AVG(e.pm25_popw) FILTER (WHERE e.year >= p.last_period_start)
                                                                    AS last_5yr
    FROM exposure AS e
    CROSS JOIN periods AS p
    WHERE e.level = 'upazila'
    GROUP BY e.pcode, e.name, e.district_pcode, e.division_pcode
)
SELECT
    b.upazila                                                       AS upazila,
    d.name                                                          AS district,
    v.name                                                          AS division,
    ROUND(b.population / 1e3)                                       AS population_k,
    ROUND(b.last_5yr, 1)                                            AS last_5yr_mean,
    ROUND(b.last_5yr - b.first_5yr, 1)                              AS change_since_first_5yr
FROM by_period AS b
JOIN areas AS d
  ON d.pcode = b.district_pcode
JOIN areas AS v
  ON v.pcode = b.division_pcode
ORDER BY b.last_5yr DESC
LIMIT 15
"""
run("q06_upazila_hotspots", q06_upazila_hotspots)

,upazila,district,division,population_k,last_5yr_mean,change_since_first_5yr
0,Savar,Dhaka,Dhaka,3651,85.3,26.2
1,Gazipur City Corporation,Gazipur,Dhaka,1853,84.6,26.4
2,Rupganj,Narayanganj,Dhaka,936,84.5,27.6
3,Kaliganj,Gazipur,Dhaka,488,84.4,25.6
4,Dhamrai,Dhaka,Dhaka,1533,84.3,24.5
5,Dhaka North City Corporation,Dhaka,Dhaka,3800,84.3,33.7
6,Kaliakair,Gazipur,Dhaka,843,84.0,24.7
7,Keraniganj,Dhaka,Dhaka,1845,83.7,29.1
8,Singair,Manikganj,Dhaka,405,83.6,24.5
9,Palash,Narsingdi,Dhaka,326,83.5,24.7


## q07. How many people live in each WHO band?

National population in each band (WHO 2021 guideline and interim targets), in millions, one row per year.

*SQL used: `PIVOT` to put the bands side by side.*

In [9]:
q07_who_bands_national = """
WITH national AS (
    SELECT
        year                                                        AS year,
        band                                                        AS band,
        ROUND(people / 1e6, 2)                                      AS people_m
    FROM who_bands
    WHERE level = 'national'
)
PIVOT national
ON band IN ('<=5 guideline', '5-10 IT-4', '10-15 IT-3', '15-25 IT-2', '25-35 IT-1', '>35 above IT-1')
USING FIRST(people_m)
GROUP BY year
ORDER BY year
"""
run("q07_who_bands_national", q07_who_bands_national)

,year,<=5 guideline,5-10 IT-4,10-15 IT-3,15-25 IT-2,25-35 IT-1,>35 above IT-1
0,1998,0,0,1.25,10.69,29.93,126.48
1,1999,0,0,0.99,6.45,28.00,132.90
2,2000,0,0,0.00,2.28,12.60,153.47
3,2001,0,0,0.00,1.21,14.31,152.83
4,2002,0,0,0.00,1.01,10.11,157.22
5,2003,0,0,0.00,0.03,9.46,158.85
6,2004,0,0,0.00,0.06,8.85,159.43
7,2005,0,0,0.00,0.42,12.45,155.48
8,2006,0,0,0.00,0.00,3.79,164.55
9,2007,0,0,0.00,0.00,4.56,163.78


## q08. How many people in each division live above 35 µg/m³?

Interim target 1 (35 µg/m³) is the most lenient WHO step. People living above it, and above IT-2 (25 µg/m³), in the latest year, by division.

*SQL used: `SUM() FILTER` on band names, a share against the division total.*

In [10]:
q08_division_above_targets = """
SELECT
    w.name                                                          AS division,
    ROUND(SUM(w.people) / 1e6, 2)                                   AS population_m,
    ROUND(SUM(w.people) FILTER (WHERE w.band = '>35 above IT-1') / 1e6, 2)
                                                                    AS above_35_m,
    ROUND(100.0 * SUM(w.people) FILTER (WHERE w.band = '>35 above IT-1') / SUM(w.people), 1)
                                                                    AS above_35_pct,
    ROUND(100.0 * SUM(w.people) FILTER (WHERE w.band IN ('25-35 IT-1', '>35 above IT-1')) / SUM(w.people), 1)
                                                                    AS above_25_pct
FROM who_bands AS w
CROSS JOIN periods AS p
WHERE w.level = 'division'
  AND w.year = p.last_year
GROUP BY w.name
ORDER BY above_35_pct DESC, population_m DESC
"""
run("q08_division_above_targets", q08_division_above_targets)

,division,population_m,above_35_m,above_35_pct,above_25_pct
0,Dhaka,45.09,45.09,100.0,100
1,Rajshahi,20.72,20.72,100.0,100
2,Rangpur,17.92,17.92,100.0,100
3,Khulna,17.76,17.76,100.0,100
4,Mymensingh,12.52,12.52,100.0,100
5,Barishal,9.29,9.29,100.0,100
6,Chattogram,33.82,33.80,99.9,100
7,Sylhet,11.22,10.22,91.1,100


## q09. Which months are worst, and does the peak month differ by division?

Average population-weighted PM2.5 for each calendar month over all years, with each division's peak and lowest month and the ratio between them.

*SQL used: `ARG_MAX` and `ARG_MIN` to name the peak and lowest month, `PIVOT` for the month columns.*

In [11]:
q09_seasonal_profile = """
WITH climatology AS (
    SELECT
        level                                                       AS level,
        name                                                        AS area,
        month                                                       AS month,
        AVG(pm25_popw)                                              AS pm25
    FROM monthly
    WHERE level IN ('national', 'division')
    GROUP BY level, name, month
),
peaks AS (
    SELECT
        area                                                        AS area,
        ARG_MAX(month, pm25)                                        AS peak_month,
        ARG_MIN(month, pm25)                                        AS lowest_month,
        ROUND(MAX(pm25) / MIN(pm25), 1)                             AS peak_to_low_ratio
    FROM climatology
    GROUP BY area
),
wide AS (
    PIVOT (SELECT area, month, ROUND(pm25, 1) AS pm25 FROM climatology)
    ON month IN (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)
    USING FIRST(pm25)
    GROUP BY area
)
SELECT
    w.*,
    p.peak_month                                                    AS peak_month,
    p.lowest_month                                                  AS lowest_month,
    p.peak_to_low_ratio                                             AS peak_to_low_ratio
FROM wide AS w
JOIN peaks AS p
  ON p.area = w.area
ORDER BY w.area = 'Bangladesh' DESC, p.peak_to_low_ratio DESC
"""
run("q09_seasonal_profile", q09_seasonal_profile)

,area,1,2,3,4,5,6,7,8,9,10,11,12,peak_month,lowest_month,peak_to_low_ratio
0,Bangladesh,121.2,101.8,73.4,43.9,35.3,30.7,26.3,28.4,28.1,52.1,83.4,109.1,1,7,4.6
1,Rangpur,125.1,111.7,71.7,47.7,35.4,25.7,20.3,23.5,29.2,48.3,75.7,105.3,1,7,6.2
2,Rajshahi,126.9,110.3,64.1,43.3,34.5,26.1,21.0,24.0,30.0,50.2,85.7,114.2,1,7,6.0
3,Mymensingh,115.0,101.7,68.5,43.3,32.2,23.1,19.6,21.8,26.3,43.6,67.8,101.1,1,7,5.9
4,Khulna,118.8,94.3,64.3,39.7,32.8,28.2,23.4,26.6,26.8,48.7,85.2,107.1,1,7,5.1
5,Barishal,120.1,85.2,67.7,37.1,31.2,33.4,31.0,33.3,26.1,56.1,101.3,120.7,12,9,4.6
6,Dhaka,139.7,114.1,84.9,48.6,41.0,36.1,31.0,33.4,31.6,63.9,99.9,130.9,1,7,4.5
7,Sylhet,76.1,67.4,50.7,35.7,28.5,19.5,17.2,18.2,22.1,31.9,41.7,58.3,1,7,4.4
8,Chattogram,109.9,95.0,80.4,43.2,34.1,35.8,31.9,32.7,25.4,50.2,77.6,96.6,1,9,4.3


## q10. Is the dry season getting worse faster than the monsoon?

Mean PM2.5 in the dry season (December to February) and the monsoon (June to August) for each year, nationally, with a straight-line slope per year for each. December is counted with the following January and February, and only complete seasons are kept. The slopes here are ordinary least squares, for a first look; the robust trend tests come in R.

*SQL used: a season `CASE`, `HAVING` to keep complete seasons, `REGR_SLOPE` as a window over the whole table.*

In [12]:
q10_dry_vs_monsoon = """
WITH seasons AS (
    SELECT
        CASE WHEN month IN (12, 1, 2) THEN dry_season_year ELSE year END
                                                                    AS season_year,
        CASE WHEN month IN (12, 1, 2) THEN 'dry (Dec-Feb)' ELSE 'monsoon (Jun-Aug)' END
                                                                    AS season,
        AVG(pm25_popw)                                              AS pm25,
        COUNT(*)                                                    AS months
    FROM monthly
    WHERE level = 'national'
      AND month IN (12, 1, 2, 6, 7, 8)
    GROUP BY 1, 2
    HAVING COUNT(*) = 3
)
SELECT
    season                                                          AS season,
    season_year                                                     AS season_year,
    ROUND(pm25, 1)                                                  AS pm25,
    ROUND(REGR_SLOPE(pm25, season_year) OVER (PARTITION BY season), 2)
                                                                    AS slope_per_year
FROM seasons
ORDER BY season, season_year
"""
run("q10_dry_vs_monsoon", q10_dry_vs_monsoon)

,season,season_year,pm25,slope_per_year
0,dry (Dec-Feb),1999,80.1,1.67
1,dry (Dec-Feb),2000,81.2,1.67
2,dry (Dec-Feb),2001,93.0,1.67
3,dry (Dec-Feb),2002,92.1,1.67
4,dry (Dec-Feb),2003,98.0,1.67
5,dry (Dec-Feb),2004,89.1,1.67
6,dry (Dec-Feb),2005,91.1,1.67
7,dry (Dec-Feb),2006,106.5,1.67
8,dry (Dec-Feb),2007,107.3,1.67
9,dry (Dec-Feb),2008,102.8,1.67


## q11. Does the 2018 jump change the national trend?

Notebook 02 found that 2018 is about 28% above the years around it in V6.GL.03 but only about 3% in the previous release, so it may be a release-specific artefact. Here the national linear slope is fitted with and without 2018. If the two slopes are close, the year does not drive the trend.

*SQL used: `REGR_SLOPE`, `REGR_R2` and `REGR_COUNT`, once on all years and once without 2018, stacked with `UNION ALL`.*

In [13]:
q11_trend_sensitivity_2018 = """
SELECT
    'all years'                                                     AS fit,
    REGR_COUNT(pm25_popw, year)                                     AS years,
    ROUND(REGR_SLOPE(pm25_popw, year), 3)                           AS slope_per_year,
    ROUND(REGR_R2(pm25_popw, year), 3)                              AS r_squared
FROM exposure
WHERE level = 'national'
UNION ALL
SELECT
    'without 2018'                                                  AS fit,
    REGR_COUNT(pm25_popw, year)                                     AS years,
    ROUND(REGR_SLOPE(pm25_popw, year), 3)                           AS slope_per_year,
    ROUND(REGR_R2(pm25_popw, year), 3)                              AS r_squared
FROM exposure
WHERE level = 'national'
  AND year <> 2018
"""
run("q11_trend_sensitivity_2018", q11_trend_sensitivity_2018)

,fit,years,slope_per_year,r_squared
0,all years,27,1.004,0.598
1,without 2018,26,0.935,0.615


## q12. Where does population weighting matter most?

The gap between population-weighted and area mean PM2.5 in the latest five years, by district. A large positive gap means people live mostly in the more polluted parts of the district (typically towns), so a simple map average would understate what residents breathe.

*SQL used: `AVG` of a difference, population density from `area_km2`, `NTILE` to split districts into quarters by density.*

In [14]:
q12_weighting_gap = """
WITH recent AS (
    SELECT
        e.name                                                      AS district,
        MAX(e.population) / MAX(e.area_km2)                         AS people_per_km2,
        AVG(e.pm25_popw)                                            AS pm25_popw,
        AVG(e.pm25_area)                                            AS pm25_area
    FROM exposure AS e
    CROSS JOIN periods AS p
    WHERE e.level = 'district'
      AND e.year >= p.last_period_start
    GROUP BY e.name
)
SELECT
    district                                                        AS district,
    ROUND(people_per_km2)                                           AS people_per_km2,
    NTILE(4) OVER (ORDER BY people_per_km2)                         AS density_quarter,
    ROUND(pm25_popw, 1)                                             AS pm25_popw,
    ROUND(pm25_area, 1)                                             AS pm25_area,
    ROUND(pm25_popw - pm25_area, 1)                                 AS weighting_gap
FROM recent
ORDER BY weighting_gap DESC
"""
run("q12_weighting_gap", q12_weighting_gap)

,district,people_per_km2,density_quarter,pm25_popw,pm25_area,weighting_gap
0,Bagerhat,405,1,70.9,65.4,5.5
1,Khulna,601,1,69.4,64.5,4.9
2,Bhola,565,1,63.1,59.1,4.0
3,Bandarban,119,1,41.0,37.2,3.9
4,Lakshmipur,1312,4,72.4,68.7,3.7
5,Rangamati,121,1,44.1,40.6,3.5
6,Noakhali,1141,3,67.6,64.1,3.5
7,Chattogram,1916,4,63.0,60.1,2.9
8,Kishoreganj,1295,3,66.7,64.6,2.2
9,Patuakhali,566,1,63.5,61.3,2.1


## Summary of saved results

In [15]:
pd.DataFrame([{"file": p.name, "rows": len(pd.read_csv(p))} for p in sorted(OUT.glob("q*.csv"))])

,file,rows
0,q01_coverage.csv,4
1,q02_national_yearly.csv,27
2,q03_period_change.csv,9
3,q04_division_ranking.csv,8
4,q05_district_extremes.csv,20
5,q06_upazila_hotspots.csv,15
6,q07_who_bands_national.csv,27
7,q08_division_above_targets.csv,8
8,q09_seasonal_profile.csv,9
9,q10_dry_vs_monsoon.csv,53


## Next: notebook 04, EDA and maps

Maps of the latest year and the change since 1998 at upazila level, division time series, the month-by-district heatmap, the WHO band picture over time, and the distribution of exposure across people.